# 06 — CNN-LSTM Baseline Training & Evaluation

**Architecture:** Zhang et al. "Sequence-to-Point Learning with Neural Networks for NILM" (AAAI 2018)  
**Model:** 5× Conv1D → center slice → Dense(1024) → Multi-task heads  
**Parameters:** 101K | INT8: 0.101 MB  

**Figures produced (publication-quality):**
1. Model architecture summary
2. Training curves (Loss / MR / F1 / MAE vs epoch)
3. Full day: Aggregate + disaggregated appliances (real timestamps)
4. Predictions vs Ground Truth with ON/OFF state overlay
5. Zoomed event detection (success + failure)
6. Confusion matrices (normalized, per appliance)
7. Error analysis: FP/FN rates + error distribution
8. Duty Cycle vs F1 scatter plot
9. Hourly error heatmap (hour of day × appliance)
10. Energy attribution: true vs predicted (pie chart)
11. Per-appliance bar charts (F1, MAE, MR)
12. Final metrics table (NILMFormer Table 2 format)

**Author:** Chadha Jeddi — NILM Benchmarking Project

## 1. Environment Setup

In [ ]:
import json, sys, os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_NILM = '/content/drive/MyDrive/nilm_project'
REPO_DIR = '/content/nilm-benchmarking'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/chadhajeddi-ux/nilm-benchmarking {REPO_DIR}

os.chdir(REPO_DIR)
for p in ['src','models','models/baselines','models/proposed']:
    sys.path.insert(0, f'{REPO_DIR}/{p}')

for d in ['data/raw/UKDALE','data/raw/REDD','data/processed']:
    os.makedirs(f'{REPO_DIR}/{d}', exist_ok=True)

links = {'data/raw/UKDALE/ukdale.h5': f'{DRIVE_NILM}/data/raw/UKDALE/ukdale.h5',
         'data/raw/REDD/redd.h5': f'{DRIVE_NILM}/data/raw/REDD/redd.h5'}
for local, remote in links.items():
    if os.path.exists(remote) and not os.path.exists(local):
        os.symlink(remote, local)

for folder in ['checkpoints','results']:
    src = f'{DRIVE_NILM}/experiments/{folder}'
    dst = f'{REPO_DIR}/experiments/{folder}'
    os.makedirs(src, exist_ok=True)
    if os.path.exists(dst) and not os.path.islink(dst):
        import shutil; shutil.rmtree(dst)
    if not os.path.islink(dst): os.symlink(src, dst)

proc_src = f'{DRIVE_NILM}/data/processed'
proc_dst = f'{REPO_DIR}/data/processed'
os.makedirs(proc_src, exist_ok=True)
if os.path.exists(proc_dst) and not os.path.islink(proc_dst):
    import shutil; shutil.rmtree(proc_dst)
if not os.path.islink(proc_dst): os.symlink(proc_src, proc_dst)

!pip install -q PyWavelets pyarrow h5py tqdm einops omegaconf torchinfo seaborn

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Working dir: {os.getcwd()}')

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import confusion_matrix
import warnings; warnings.filterwarnings('ignore')

from config import WINDOW_SIZE, INPUT_CHANNELS, N_APPLIANCES, APPLIANCE_NAMES, APPLIANCES, SEED
from preprocessing import load_ukdale_house, preprocess_house
from dataset import NILMDataset, NormStats, split_train_val, build_dataloaders, load_clean_df, save_clean_df
from metrics import MetricsTracker, multi_task_loss
from train import train_one_epoch, validate_one_epoch, EarlyStopping

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MODEL-SPECIFIC — change only these 3 lines
from cnn_lstm_model import CNNLSTMBaseline as ModelClass
MODEL_NAME = 'cnn_lstm'
LR = 1e-3
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 100
PATIENCE = 15
COLORS = {'kettle':'#D94040','fridge':'#2E9E5A','washing_machine':'#E8922A',
          'dishwasher':'#7B4FBF','microwave':'#CC3399'}
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 10
print(f'Model: {MODEL_NAME} | Device: {DEVICE} | Epochs: {EPOCHS}')

## 2. Data Loading & Statistics

In [ ]:
cached = load_clean_df('UK-DALE', 1)
if cached is not None: clean_df = cached
else:
    raw_df = load_ukdale_house(house=1)
    clean_df = preprocess_house(raw_df)
    save_clean_df(clean_df, 'UK-DALE', 1)

print(f'Shape: {clean_df.shape} | Duration: {(clean_df.index[-1]-clean_df.index[0]).days} days')

# Dataset statistics table
stats_rows = []
for a in APPLIANCE_NAMES:
    state = clean_df[f'{a}_state']
    power = clean_df[a]
    stats_rows.append({'Appliance': a,
        'Threshold (W)': APPLIANCES[a]['power_threshold'],
        'Max Power (W)': APPLIANCES[a]['max_power'],
        'Mean ON Power (W)': f"{power[power>APPLIANCES[a]['power_threshold']].mean():.1f}",
        'Duty Cycle (%)': f'{state.mean()*100:.2f}',
        'ON Events': int((state.diff()==1).sum())})
pd.DataFrame(stats_rows)

In [ ]:
train_df, val_df = split_train_val(clean_df, val_fraction=0.15)
train_loader, val_loader, norm_stats = build_dataloaders(
    train_df, val_df, batch_size=256, train_stride=120,
    val_stride=480, num_workers=2, add_temporal_features=True)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')
x, yp, ys = next(iter(val_loader))
print(f'Batch: x={tuple(x.shape)}, y_power={tuple(yp.shape)}')

## 3. Power Signatures & Daily Patterns

In [ ]:
# Figure 1: Characteristic power signature per appliance
# Shows the ON-state power shape — justifies why some appliances are harder to detect
fig, axes = plt.subplots(1, N_APPLIANCES, figsize=(18, 4))
for i, a in enumerate(APPLIANCE_NAMES):
    state = clean_df[f'{a}_state']
    power = clean_df[a]
    # Find first ON event and extract 60 minutes around it
    on_starts = clean_df.index[(state.diff()==1)]
    if len(on_starts) > 5:
        event_time = on_starts[5]
        event_pos = clean_df.index.get_loc(event_time)
        s = max(0, event_pos-30); e = min(len(clean_df), event_pos+570)
        segment = power.iloc[s:e].values
        axes[i].plot(segment, color=COLORS[a], linewidth=1.5)
        axes[i].axvline(30, color='black', linestyle=':', alpha=0.5, label='Turn-ON')
        axes[i].fill_between(range(len(segment)), segment, alpha=0.2, color=COLORS[a])
    axes[i].set_title(a.replace('_',' ').title(), fontsize=11)
    axes[i].set_xlabel('Time (×6s)')
    if i==0: axes[i].set_ylabel('Power (W)')
    duty = clean_df[f'{a}_state'].mean()*100
    axes[i].text(0.05, 0.92, f'Duty: {duty:.1f}%', transform=axes[i].transAxes,
                 fontsize=9, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.suptitle('Characteristic Power Signatures per Appliance\n(vertical line = turn-ON moment)', fontsize=13)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_power_signatures.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Full day view — aggregate + all appliances (real timestamps)
# Standard NILM visualization from Kelly & Knottenbelt 2015
day_str = '2013-06-15'
day_data = clean_df.loc[day_str] if day_str in str(clean_df.index[1000]) else clean_df.iloc[10000:14400]

fig, axes = plt.subplots(N_APPLIANCES+1, 1, figsize=(16, 12), sharex=True)

axes[0].plot(day_data.index, day_data['aggregate'], color='#3366CC', linewidth=0.8)
axes[0].fill_between(day_data.index, day_data['aggregate'], alpha=0.2, color='#3366CC')
axes[0].set_ylabel('Aggregate\n(W)', fontsize=10)
axes[0].set_title('Full Day Energy Disaggregation — UK-DALE House 1', fontsize=13)

for i, a in enumerate(APPLIANCE_NAMES):
    ax = axes[i+1]
    ax.plot(day_data.index, day_data[a], color=COLORS[a], linewidth=0.8)
    ax.fill_between(day_data.index, day_data[a], alpha=0.25, color=COLORS[a])
    ax.set_ylabel(f'{a.replace("_"," ").title()}\n(W)', fontsize=9)
    duty = day_data[f'{a}_state'].mean()*100
    ax.text(0.01, 0.85, f'Duty: {duty:.1f}%', transform=ax.transAxes,
            fontsize=8, bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].set_xlabel('Time of Day')
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_full_day.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Architecture

In [ ]:
from torchinfo import summary
model = ModelClass(in_channels=INPUT_CHANNELS, window_size=WINDOW_SIZE, n_appliances=N_APPLIANCES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,} | INT8: {n_params/1e6:.3f} MB')
summary(model, input_size=(1, INPUT_CHANNELS, WINDOW_SIZE),
        col_names=['input_size','output_size','num_params'], depth=3)

## 5. Training (100 epochs)

In [ ]:
import time
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
early_stop = EarlyStopping(patience=PATIENCE)
app_max = {a: float(APPLIANCES[a]['max_power']) for a in APPLIANCE_NAMES}
tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
history = {'epoch':[],'train_loss':[],'val_loss':[],'val_mr':[],'val_f1':[],'val_mae':[],'lr':[]}
best_mr, best_state, best_epoch = -float('inf'), None, 0
start_time = time.time()

for epoch in range(1, EPOCHS+1):
    ep_start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss, metrics = validate_one_epoch(model, val_loader, DEVICE, tracker)
    val_mr=metrics['mean']['mr']; val_f1=metrics['mean']['f1']; val_mae=metrics['mean']['mae_w']
    cur_lr = optimizer.param_groups[0]['lr']
    for k,v in zip(['epoch','train_loss','val_loss','val_mr','val_f1','val_mae','lr'],
                   [epoch,train_loss,val_loss,val_mr,val_f1,val_mae,cur_lr]):
        history[k].append(v)
    is_best = val_mr > best_mr
    if is_best:
        best_mr=val_mr; best_epoch=epoch
        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        # Auto-save checkpoint every MR improvement
        if is_best:
            torch.save({
                'model_name': MODEL_NAME,
                'model_state_dict': best_state,
                'best_epoch': best_epoch,
                'best_val_mr': best_mr,
                'n_params': n_params,
                'norm_stats': {
                    'agg_mean': norm_stats.agg_mean,
                    'agg_std': norm_stats.agg_std,
                    'appliance_max': norm_stats.appliance_max,
                },
            }, f'experiments/checkpoints/{MODEL_NAME}_best.pth')

        # Auto-save history every 5 epochs
        if epoch % 5 == 0 or is_best:
            import json as _json
            _hist = {**history,
                'best_epoch': best_epoch,
                'best_val_mr': best_mr,
                'model': MODEL_NAME}
            with open(f'experiments/results/{MODEL_NAME}_history.json','w') as _f:
                _json.dump(_hist, _f)

    print(f"Ep {epoch:3d}/{EPOCHS} | L:{train_loss:.4f}/{val_loss:.4f} | "
          f"MR:{val_mr:.3f} F1:{val_f1:.3f} MAE:{val_mae:.1f}W | "
          f"LR:{cur_lr:.1e} | {time.time()-ep_start:.0f}s{'★' if is_best else ''}")
    scheduler.step(val_mr)
    if early_stop.step(val_mr): print(f'Early stopping at epoch {epoch}'); break

total_time = time.time()-start_time
print(f'\nDone in {total_time/60:.1f} min | Best epoch: {best_epoch} | Best MR: {best_mr:.4f}')

## Reload from Drive
**Run this cell if the session restarted** — restores all variables without retraining.

In [ ]:
# Reload from Drive if session restarted
import json, os, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import confusion_matrix
from config import INPUT_CHANNELS, WINDOW_SIZE, N_APPLIANCES, APPLIANCE_NAMES, APPLIANCES
from metrics import MetricsTracker
from train import validate_one_epoch, get_model_registry
from dataset import load_clean_df, split_train_val, build_dataloaders

COLORS = {'kettle':'#D94040','fridge':'#2E9E5A',
          'washing_machine':'#E8922A','dishwasher':'#7B4FBF','microwave':'#CC3399'}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# 1. Reload history
hist_path = f'experiments/results/{MODEL_NAME}_history.json'
assert os.path.exists(hist_path), f'No history at {hist_path} - run training first'
with open(hist_path) as f:
    history = json.load(f)
best_epoch = history['best_epoch']
best_mr = history['best_val_mr']
total_time = history.get('training_time_seconds', 0)
print(f'History: {len(history["epoch"])} epochs | Best MR: {best_mr:.4f} @ ep{best_epoch}')

# 2. Reload model
ckpt_path = f'experiments/checkpoints/{MODEL_NAME}_best.pth'
assert os.path.exists(ckpt_path), f'No checkpoint at {ckpt_path} - run training first'
registry = get_model_registry()
model = registry[MODEL_NAME]().to(DEVICE)
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME} | {n_params:,} params | epoch {ckpt["best_epoch"]}')

# 3. Reload data
from preprocessing import load_ukdale_house, preprocess_house
from dataset import save_clean_df
cached = load_clean_df('UK-DALE', 1)
if cached is not None:
    clean_df = cached
else:
    raw_df = load_ukdale_house(house=1)
    clean_df = preprocess_house(raw_df)
    save_clean_df(clean_df, 'UK-DALE', 1)
train_df, val_df = split_train_val(clean_df, val_fraction=0.15)
_, val_loader, norm_stats = build_dataloaders(
    train_df, val_df, batch_size=256, train_stride=120,
    val_stride=480, num_workers=2, add_temporal_features=True)
print(f'Data: {len(val_df):,} val rows')

# 4. Collect predictions
app_max = {a: float(APPLIANCES[a]['max_power']) for a in APPLIANCE_NAMES}
tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
_, final_metrics = validate_one_epoch(model, val_loader, DEVICE, tracker)
all_pp, all_tp, all_ps, all_ts = [], [], [], []
with torch.no_grad():
    for x, yp, ys in val_loader:
        pp, ps, pg = model(x.to(DEVICE))
        all_pp.append(pp.cpu().numpy())
        all_tp.append(yp.numpy())
        all_ps.append((torch.sigmoid(ps)>=0.5).float().cpu().numpy())
        all_ts.append(ys.numpy())
pred_power = np.concatenate(all_pp)
true_power = np.concatenate(all_tp)
pred_state = np.concatenate(all_ps)
true_state = np.concatenate(all_ts)
print(f'Predictions: {pred_power.shape[0]:,} samples')
print('\nFinal metrics:')
tracker.print_table(final_metrics)


## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
epochs = history['epoch']
configs = [
    (axes[0,0], 'Loss', None, [('train_loss','#3366CC','Train'), ('val_loss','#D94040','Val')]),
    (axes[0,1], 'Matching Ratio ↑ (checkpoint metric)', '#2E9E5A', [('val_mr','#2E9E5A','MR')]),
    (axes[1,0], 'F1 Score ↑', '#E8922A', [('val_f1','#E8922A','F1')]),
    (axes[1,1], 'MAE (Watts) ↓', '#7B4FBF', [('val_mae','#7B4FBF','MAE')]),
]
for ax, title, _, series in configs:
    for key, color, label in series:
        ax.plot(epochs, history[key], color=color, linewidth=2, label=label)
    ax.axvline(best_epoch, color='gray', linestyle=':', alpha=0.7, label=f'Best ep{best_epoch}')
    ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')
plt.suptitle(f'{MODEL_NAME.upper()} Baseline — Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation & Predictions

In [ ]:
# Load best model and collect predictions
model.load_state_dict(best_state); model.to(DEVICE); model.eval()
_, final_metrics = validate_one_epoch(model, val_loader, DEVICE, tracker)
print(f'\n{"="*80}\nFINAL RESULTS — {MODEL_NAME.upper()} (epoch {best_epoch})\n{"="*80}')
tracker.print_table(final_metrics)

all_pp,all_tp,all_ps,all_ts = [],[],[],[]
with torch.no_grad():
    for x,yp,ys in val_loader:
        pp,ps,pg = model(x.to(DEVICE))
        all_pp.append(pp.cpu().numpy()); all_tp.append(yp.numpy())
        all_ps.append((torch.sigmoid(ps)>=0.5).float().cpu().numpy()); all_ts.append(ys.numpy())
pred_power=np.concatenate(all_pp); true_power=np.concatenate(all_tp)
pred_state=np.concatenate(all_ps); true_state=np.concatenate(all_ts)
print(f'Predictions: {pred_power.shape[0]:,} samples collected')

In [ ]:
# Figure 3: Predictions vs Ground Truth with ON/OFF state overlay
# Inspired by Kelly & Knottenbelt 2015 and NILMFormer 2025
N_SHOW, start_idx = 600, len(pred_power)//3
fig, axes = plt.subplots(N_APPLIANCES, 1, figsize=(16, 3*N_APPLIANCES), sharex=True)
for i, a in enumerate(APPLIANCE_NAMES):
    ax = axes[i]
    max_w = APPLIANCES[a]['max_power']
    tw = true_power[start_idx:start_idx+N_SHOW,i]*max_w
    pw = pred_power[start_idx:start_idx+N_SHOW,i]*max_w
    ts = true_state[start_idx:start_idx+N_SHOW,i]
    ps = pred_state[start_idx:start_idx+N_SHOW,i]
    t = np.arange(N_SHOW)

    ax.plot(t, tw, color=COLORS[a], alpha=0.8, linewidth=1.2, label='Ground Truth')
    ax.plot(t, pw, color='#222', alpha=0.6, linewidth=0.8, linestyle='--', label='Predicted')

    # ON/OFF state overlay: green=correct ON, red=FP, blue=FN
    for t_idx in range(N_SHOW):
        if ts[t_idx]==1 and ps[t_idx]==1:   # True positive
            ax.axvspan(t_idx, t_idx+1, alpha=0.15, color='green')
        elif ts[t_idx]==0 and ps[t_idx]==1: # False positive
            ax.axvspan(t_idx, t_idx+1, alpha=0.25, color='red')
        elif ts[t_idx]==1 and ps[t_idx]==0: # False negative
            ax.axvspan(t_idx, t_idx+1, alpha=0.25, color='blue')

    ax.set_ylabel(f'{a.replace("_",chr(10)).title()}\n(W)', fontsize=9)
    mae_val = final_metrics[a]['mae_w']
    f1_val = final_metrics[a]['f1']
    ax.text(0.01, 0.87, f'MAE={mae_val:.1f}W | F1={f1_val:.3f}',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    if i==0:
        ax.legend(loc='upper right', fontsize=8)
        patches = [mpatches.Patch(color='green',alpha=0.4,label='True Positive'),
                   mpatches.Patch(color='red',alpha=0.4,label='False Positive'),
                   mpatches.Patch(color='blue',alpha=0.4,label='False Negative')]
        ax.legend(handles=patches+ax.get_lines()[:2], loc='upper right', fontsize=7)

axes[-1].set_xlabel('Timestep (×6 seconds)')
plt.suptitle(f'{MODEL_NAME.upper()} — Predictions vs Ground Truth with ON/OFF State Overlay',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: Zoomed event detection — success and failure cases
# Inspired by Zhang et al. AAAI 2018 Figure 3
fig, axes = plt.subplots(2, N_APPLIANCES, figsize=(18, 8))
ZOOM = 200  # 200 timesteps = 20 minutes

for i, a in enumerate(APPLIANCE_NAMES):
    max_w = APPLIANCES[a]['max_power']
    ts_all = true_state[:, i]; ps_all = pred_state[:, i]
    tw_all = true_power[:, i]*max_w; pw_all = pred_power[:, i]*max_w

    # Find a SUCCESS case (TP: model correctly detected ON)
    tp_mask = (ts_all==1) & (ps_all==1)
    success_idx = np.where(tp_mask)[0]
    if len(success_idx) > ZOOM:
        s = max(0, success_idx[len(success_idx)//2]-ZOOM//2)
        axes[0,i].plot(tw_all[s:s+ZOOM], color=COLORS[a], linewidth=1.5, label='Truth')
        axes[0,i].plot(pw_all[s:s+ZOOM], color='#333', linewidth=1, linestyle='--', label='Pred')
        axes[0,i].set_title(f'{a.replace("_"," ").title()}\n✅ Success', fontsize=10, color='green')
    else:
        axes[0,i].text(0.5, 0.5, 'No TP found', ha='center', va='center', transform=axes[0,i].transAxes)
        axes[0,i].set_title(f'{a.replace("_"," ").title()}\n(no events)', fontsize=10)

    # Find a FAILURE case (FN: model missed ON event)
    fn_mask = (ts_all==1) & (ps_all==0)
    fail_idx = np.where(fn_mask)[0]
    if len(fail_idx) > ZOOM:
        s = max(0, fail_idx[len(fail_idx)//2]-ZOOM//2)
        axes[1,i].plot(tw_all[s:s+ZOOM], color=COLORS[a], linewidth=1.5, label='Truth')
        axes[1,i].plot(pw_all[s:s+ZOOM], color='#333', linewidth=1, linestyle='--', label='Pred')
        axes[1,i].set_title(f'❌ Missed Event (FN)', fontsize=10, color='red')
    else:
        axes[1,i].text(0.5, 0.5, 'No FN found\n(good!)', ha='center', va='center',
                       transform=axes[1,i].transAxes, color='green')
        axes[1,i].set_title('No missed events', fontsize=10, color='green')

    for row in [0,1]:
        axes[row,i].set_xlabel('Timestep (×6s)')
        if i==0: axes[row,i].set_ylabel('Power (W)')

handles = [plt.Line2D([0],[0],color=COLORS[APPLIANCE_NAMES[0]],label='Ground Truth'),
           plt.Line2D([0],[0],color='#333',linestyle='--',label='Predicted')]
fig.legend(handles=handles, loc='upper center', ncol=2, fontsize=10)
plt.suptitle(f'{MODEL_NAME.upper()} — Zoomed Event Detection: Success vs Failure', fontsize=13, y=1.03)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_zoomed_events.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, N_APPLIANCES, figsize=(4*N_APPLIANCES, 4))
for i, a in enumerate(APPLIANCE_NAMES):
    cm = confusion_matrix(true_state[:,i], pred_state[:,i], labels=[0,1])
    cm_norm = cm.astype('float')/(cm.sum(axis=1,keepdims=True)+1e-8)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=['OFF','ON'], yticklabels=['OFF','ON'],
                ax=axes[i], cbar=i==N_APPLIANCES-1, vmin=0, vmax=1)
    prec = final_metrics[a]['precision']; rec = final_metrics[a]['recall']
    axes[i].set_title(f'{a.replace("_"," ").title()}\nF1={final_metrics[a]["f1"]:.3f} P={prec:.3f} R={rec:.3f}',
                      fontsize=9)
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('Actual' if i==0 else '')
plt.suptitle(f'{MODEL_NAME.upper()} — Normalized Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Error Analysis

In [ ]:
# Figure: FP/FN rates + Error distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fp_rates=[]; fn_rates=[]
for i, a in enumerate(APPLIANCE_NAMES):
    ts,ps = true_state[:,i], pred_state[:,i]
    fp_rates.append(((ts==0)&(ps==1)).sum()/max((ts==0).sum(),1))
    fn_rates.append(((ts==1)&(ps==0)).sum()/max((ts==1).sum(),1))
x_pos=np.arange(N_APPLIANCES); w=0.35
axes[0].bar(x_pos-w/2, fp_rates, w, label='False Positive Rate', color='#E8922A', alpha=0.8)
axes[0].bar(x_pos+w/2, fn_rates, w, label='False Negative Rate', color='#3366CC', alpha=0.8)
axes[0].set_xticks(x_pos); axes[0].set_xticklabels(APPLIANCE_NAMES, rotation=20)
axes[0].set_ylabel('Rate'); axes[0].set_title('Error Types per Appliance'); axes[0].legend()
for i,a in enumerate(APPLIANCE_NAMES):
    errors = (pred_power[:,i]-true_power[:,i])*APPLIANCES[a]['max_power']
    axes[1].hist(errors, bins=60, alpha=0.5, label=a.replace('_',' '), color=COLORS[a])
axes[1].axvline(0, color='black', linewidth=1.5, label='Perfect (0W error)')
axes[1].set_xlabel('Power Error (W)'); axes[1].set_ylabel('Count')
axes[1].set_title('Power Error Distribution'); axes[1].legend(fontsize=8); axes[1].set_xlim(-300,300)
plt.suptitle(f'{MODEL_NAME.upper()} — Error Analysis', fontsize=13)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Duty Cycle vs F1 Scatter

In [ ]:
# Key analytical figure: explains WHY some appliances are harder to detect
# Inspired by Virtsionis et al. SAED (Machine Learning 2021)
fig, ax = plt.subplots(figsize=(8, 6))
duty_cycles = [clean_df[f'{a}_state'].mean()*100 for a in APPLIANCE_NAMES]
f1_scores = [final_metrics[a]['f1'] for a in APPLIANCE_NAMES]
mae_scores = [final_metrics[a]['mae_w'] for a in APPLIANCE_NAMES]

for i, a in enumerate(APPLIANCE_NAMES):
    ax.scatter(duty_cycles[i], f1_scores[i], s=mae_scores[i]*2,
               color=COLORS[a], alpha=0.8, zorder=5)
    ax.annotate(a.replace('_','\n'), (duty_cycles[i], f1_scores[i]),
                textcoords='offset points', xytext=(8,5), fontsize=9)

ax.set_xlabel('Duty Cycle (% of time ON)', fontsize=12)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title(f'{MODEL_NAME.upper()} — Duty Cycle vs F1 Score\n(bubble size = MAE in Watts)', fontsize=12)
ax.set_xlim(-2, 55); ax.set_ylim(-0.05, 1.05)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='F1=0.5 threshold')
ax.legend(fontsize=9)
# Add annotation explaining the trend
ax.text(0.02, 0.02,
    'Higher duty cycle → more ON examples → easier to detect\n'
    'Fridge (43%) is easiest; Kettle (0.6%) is hardest',
    transform=ax.transAxes, fontsize=9,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_duty_vs_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Hourly Error Heatmap

In [ ]:
# Shows WHEN the model makes errors — motivates temporal features
# New figure inspired by recent NILM analysis papers
val_index = val_df.index[WINDOW_SIZE//2: WINDOW_SIZE//2 + len(pred_power)]
if len(val_index) > len(pred_power):
    val_index = val_index[:len(pred_power)]
pred_p_cut = pred_power[:len(val_index)]
true_p_cut = true_power[:len(val_index)]

hours = pd.Series(val_index).dt.hour.values
heatmap_data = np.zeros((24, N_APPLIANCES))
for h in range(24):
    mask = hours == h
    if mask.sum() > 0:
        for i, a in enumerate(APPLIANCE_NAMES):
            max_w = APPLIANCES[a]['max_power']
            heatmap_data[h, i] = np.mean(np.abs(
                pred_p_cut[mask,i]*max_w - true_p_cut[mask,i]*max_w))

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heatmap_data, xticklabels=[a.replace('_','\n') for a in APPLIANCE_NAMES],
            yticklabels=[f'{h:02d}:00' for h in range(24)],
            cmap='YlOrRd', ax=ax, fmt='.0f', annot=True,
            cbar_kws={'label': 'MAE (Watts)'})
ax.set_xlabel('Appliance', fontsize=12)
ax.set_ylabel('Hour of Day', fontsize=12)
ax.set_title(f'{MODEL_NAME.upper()} — Hourly Error Heatmap (MAE in W per hour)\n'
             f'Dark = high error | Motivates temporal feature injection', fontsize=12)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_hourly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Energy Attribution

In [ ]:
# Figure: True vs predicted total energy per appliance
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
true_energy = [true_power[:,i].sum()*APPLIANCES[a]['max_power'] for i,a in enumerate(APPLIANCE_NAMES)]
pred_energy = [pred_power[:,i].sum()*APPLIANCES[a]['max_power'] for i,a in enumerate(APPLIANCE_NAMES)]
colors_list = [COLORS[a] for a in APPLIANCE_NAMES]
labels = [a.replace('_','\n') for a in APPLIANCE_NAMES]

# Pie charts
axes[0].pie(true_energy, labels=labels, colors=colors_list, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize':9})
axes[0].set_title('True Energy Distribution', fontsize=12)

axes[1].pie(pred_energy, labels=labels, colors=colors_list, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize':9})
axes[1].set_title('Predicted Energy Distribution', fontsize=12)

# Add SAE annotation
sae_vals = [final_metrics[a]['sae'] for a in APPLIANCE_NAMES]
mean_sae = np.mean(sae_vals)
fig.text(0.5, 0.02, f'Mean SAE = {mean_sae:.3f} (0=perfect, 1=100% error)',
         ha='center', fontsize=11, style='italic')
plt.suptitle(f'{MODEL_NAME.upper()} — Energy Attribution: True vs Predicted', fontsize=13)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_energy.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Per-Appliance Bar Charts

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colors = [COLORS[a] for a in APPLIANCE_NAMES]
x_pos = np.arange(N_APPLIANCES)
metrics_to_plot = [
    ('f1','F1 Score ↑','{:.3f}',True),
    ('mae_w','MAE (W) ↓','{:.1f}',False),
    ('mr','Matching Ratio ↑','{:.3f}',True),
    ('precision','Precision ↑','{:.3f}',True),
    ('recall','Recall ↑','{:.3f}',True),
    ('teca','TECA ↑','{:.3f}',True),
]
for ax, (metric, label, fmt, higher_better) in zip(axes, metrics_to_plot):
    vals = [final_metrics[a][metric] for a in APPLIANCE_NAMES]
    bars = ax.bar(x_pos, vals, color=colors, alpha=0.85)
    ax.set_xticks(x_pos); ax.set_xticklabels(APPLIANCE_NAMES, rotation=25, ha='right')
    ax.set_title(label, fontsize=11)
    best_idx = np.argmax(vals) if higher_better else np.argmin(vals)
    bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val*1.02, fmt.format(val),
                ha='center', fontsize=8)
plt.suptitle(f'{MODEL_NAME.upper()} Baseline — All Metrics per Appliance\n(gold border = best appliance)',
             fontsize=13)
plt.tight_layout()
plt.savefig(f'experiments/results/{MODEL_NAME}_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Save All Results to Drive

In [ ]:
import json
# Checkpoint
checkpoint = {'model_name':MODEL_NAME,'model_state_dict':best_state,
    'best_epoch':best_epoch,'best_val_mr':best_mr,'n_params':n_params,
    'norm_stats':{'agg_mean':norm_stats.agg_mean,'agg_std':norm_stats.agg_std,
                  'appliance_max':norm_stats.appliance_max}}
torch.save(checkpoint, f'experiments/checkpoints/{MODEL_NAME}_best.pth')
# History
history.update({'model':MODEL_NAME,'training_time_seconds':total_time,
                'best_epoch':best_epoch,'best_val_mr':best_mr})
with open(f'experiments/results/{MODEL_NAME}_history.json','w') as f: json.dump(history,f,indent=2)
# Metrics
tracker.save_json(f'experiments/results/{MODEL_NAME}_metrics.json',final_metrics,model_name=MODEL_NAME)
tracker.to_dataframe(final_metrics).to_csv(f'experiments/results/{MODEL_NAME}_metrics.csv',index=False)
norm_stats.save(f'experiments/results/{MODEL_NAME}_norm_stats.json')

print('✅ ALL RESULTS SAVED TO GOOGLE DRIVE')
print(f'   Model: {MODEL_NAME.upper()}')
print(f'   Best epoch: {best_epoch} | Best MR: {best_mr:.4f}')
print(f'   Training time: {total_time/60:.1f} minutes')
print(f'\nFigures saved:')
figs = ['power_signatures','full_day','training_curves','predictions',
        'zoomed_events','confusion','error_analysis','duty_vs_f1',
        'hourly_heatmap','energy','bar_charts']
for f in figs:
    print(f'   experiments/results/{MODEL_NAME}_{f}.png')